# A4: Sensitivity Analysis — Coop-DABC vs Coop-SAC
### Appendix 대응: Reviewer R2.2 (Degradation) + Imbalance Tolerance

## 분석 목적
두 파라미터를 1D 스윕으로 변화시켜 **Coop-DABC가 Coop-SAC 대비 우월성을 파라미터 설정과 무관하게 유지하는지** 확인합니다.

| Appendix | 고정 | 변화 | 시나리오 수 |
|----------|------|------|-------------|
| **A (Degradation)** | tol = 5.0 MW | deg_cost = 2.5 / 5.0 / 7.5 / 10.0 $/MWh | 4 |
| **B (Tolerance)**   | deg = 5.0 $/MWh | tol = 2.0 / 5.0 / 10.0 MW | 3 |
| 중복 baseline 제외 | | **총 6 시나리오** | |

## 알고리즘
- **Coop-DABC**: MILP expert CSV → BC imitation + TD3 RL  
- **Coop-SAC**: pure RL (expert 불필요)

## Implication
> "어떤 파라미터 설정에서도 Coop-DABC가 Coop-SAC보다 일관되게 우월하며, 두 모델 모두 파라미터 변화에 robust하다."

## 실행 순서
| Step | 내용 | 비고 |
|------|------|------|
| **Config** | 항상 먼저 실행 | |
| **Step 1** | MILP 6회 → expert CSV 생성 | ~15–30분 |
| **Step 2** | expert CSV 확인 | 즉시 |
| **Step 3** | Coop-DABC 학습 (6 × N seeds) | 수 시간 |
| **Step 4** | Coop-SAC 학습 (6 × N seeds) | 수 시간 |
| **Step 5** | 실행 로그 저장 | 즉시 |
| **Step 6** | 결과 수집 및 비교표 | 즉시 |
| **Step 7** | 시각화 및 CSV 저장 | 즉시 |

In [ ]:
import subprocess, sys, os, time, json, glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

CODE_DIR    = os.path.abspath('../code')
RESULTS_DIR = os.path.abspath('../results/sensitivity')
EXPERT_DIR  = os.path.abspath('../data/processed/expert_actions')
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── 시나리오 정의 ─────────────────────────────────────────────────────────
SCEN_DEG = [(2.5, 5.0), (5.0, 5.0), (7.5, 5.0), (10.0, 5.0)]   # Appendix A
SCEN_TOL = [(5.0, 2.0), (5.0, 5.0), (5.0, 10.0)]                # Appendix B
ALL_SCENARIOS = SCEN_DEG + [(d, t) for d, t in SCEN_TOL if (d, t) not in SCEN_DEG]

SEEDS = [0, 1, 2, 3, 4]   # 5 seeds (baseline과 일치)

def expert_csv_path(cost, tol):
    return os.path.join(EXPERT_DIR, f'Offline_Expert_Action_joint_deg{cost}_tol{tol}.csv')

def result_json_path(algo, cost, tol, seed):
    return os.path.join(RESULTS_DIR, f'result_{algo}_deg{cost}_tol{tol}_seed{seed}.json')

def run_cmd(args_list, label='', cwd=CODE_DIR):
    """subprocess 실행. 출력은 캡처하고 실패 시에만 표시 (flood 방지)."""
    t0 = time.time()
    print(f'[RUN] {label}', flush=True)
    proc = subprocess.run(
        [sys.executable] + [str(a) for a in args_list],
        cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
    )
    elapsed = time.time() - t0
    if proc.returncode == 0:
        print(f'  done in {elapsed:.0f}s', flush=True)
    else:
        print(f'  [FAILED exit {proc.returncode}]  {elapsed:.0f}s', flush=True)
        print(proc.stdout[-3000:])
    return proc.returncode == 0

print(f'CODE_DIR     : {CODE_DIR}')
print(f'RESULTS_DIR  : {RESULTS_DIR}')
print(f'총 시나리오   : {len(ALL_SCENARIOS)}  /  SEEDS: {SEEDS}')
print('(5,5)는 baseline 결과 재사용 → 학습 시 자동 SKIP')
print()
print('시나리오 목록:')
for i, (d, t) in enumerate(ALL_SCENARIOS):
    tag = ' ★ baseline(공유)' if (d, t) == (5.0, 5.0) else ''
    print(f'  {i+1}. deg={d} $/MWh  tol={t} MW{tag}')

---
## Step 1: MILP 최적화 — Expert CSV 생성 (6회)

In [ ]:
milp_log = []
for cost, tol in ALL_SCENARIOS:
    csv_path = expert_csv_path(cost, tol)

    # ── 이미 파일이 있으면 건너뜀 ──────────────────────────────────────────
    if os.path.exists(csv_path):
        df_chk = pd.read_csv(csv_path)
        print(f'[SKIP] deg={cost}  tol={tol}  →  {os.path.basename(csv_path)} 이미 존재 ({len(df_chk)}행)')
        milp_log.append({'deg_cost': cost, 'tol': tol, 'ok': True, 'skipped': True})
        continue

    ok = run_cmd(
        ['optimization/run_optimization.py', '--mode', 'joint',
         '--deg_cost', cost, '--tol', tol],
        label=f'MILP  deg={cost}$/MWh  tol={tol}MW'
    )
    milp_log.append({'deg_cost': cost, 'tol': tol, 'ok': ok, 'skipped': False})

print('\n=== MILP 완료 ===')
for r in milp_log:
    flag = '(skipped)' if r.get('skipped') else ''
    print(f"  deg={r['deg_cost']}  tol={r['tol']}  {'OK' if r['ok'] else 'FAILED'}  {flag}")

---
## Step 2: Expert CSV 확인

In [ ]:
print(f'{"deg":>6}  {"tol":>5}  {"rows":>7}  {"bid_mean":>9}  {"ope_mean":>9}  status')
print('-' * 60)
missing = []
for cost, tol in ALL_SCENARIOS:
    path = expert_csv_path(cost, tol)
    if os.path.exists(path):
        df = pd.read_csv(path)
        print(f'{cost:>6}  {tol:>5}  {len(df):>7}  '
              f'{df.Bid_Action.mean():>9.3f}  {df.Ope_Action.mean():>9.3f}  OK')
    else:
        print(f'{cost:>6}  {tol:>5}  MISSING')
        missing.append((cost, tol))

if missing:
    print(f'\nWARNING: {len(missing)}개 누락 — Step 1 먼저 실행 필요')
else:
    print('\nAll expert CSVs OK')

---
## Step 3: Coop-DABC 학습 (6 시나리오 × N seeds)

In [ ]:
dabc_log = []
for cost, tol in ALL_SCENARIOS:
    csv = expert_csv_path(cost, tol)
    if not os.path.exists(csv):
        print(f'SKIP deg={cost} tol={tol}: expert CSV 없음')
        continue
    for seed in SEEDS:
        rp = result_json_path('coop_bc', cost, tol, seed)
        if os.path.exists(rp):
            print(f'[SKIP] Coop-DABC  deg={cost} tol={tol} seed={seed} (결과 존재)')
            dabc_log.append({'algo': 'coop_dabc', 'deg_cost': cost, 'tol': tol, 'seed': seed, 'ok': True})
            continue
        ok = run_cmd(
            ['run_training.py', '--algo', 'coop_bc',
             '--deg_cost', cost, '--tol', tol, '--seed', seed,
             '--expert_csv', csv],
            label=f'Coop-DABC  deg={cost}  tol={tol}  seed={seed}'
        )
        dabc_log.append({'algo': 'coop_dabc', 'deg_cost': cost, 'tol': tol, 'seed': seed, 'ok': ok})

print('\n=== Coop-DABC 완료 ===')
df_dabc = pd.DataFrame(dabc_log)
if not df_dabc.empty:
    print(df_dabc.groupby(['deg_cost', 'tol'])['ok'].all().unstack('tol').to_string())

---
## Step 4: Coop-SAC 학습 (6 시나리오 × N seeds)

In [ ]:
sac_log = []
for cost, tol in ALL_SCENARIOS:
    for seed in SEEDS:
        rp = result_json_path('coop_sac', cost, tol, seed)
        if os.path.exists(rp):
            print(f'[SKIP] Coop-SAC  deg={cost} tol={tol} seed={seed} (결과 존재)')
            sac_log.append({'algo': 'coop_sac', 'deg_cost': cost, 'tol': tol, 'seed': seed, 'ok': True})
            continue
        ok = run_cmd(
            ['run_training.py', '--algo', 'coop_sac',
             '--deg_cost', cost, '--tol', tol, '--seed', seed],
            label=f'Coop-SAC  deg={cost}  tol={tol}  seed={seed}'
        )
        sac_log.append({'algo': 'coop_sac', 'deg_cost': cost, 'tol': tol, 'seed': seed, 'ok': ok})

print('\n=== Coop-SAC 완료 ===')
df_sac = pd.DataFrame(sac_log)
if not df_sac.empty:
    print(df_sac.groupby(['deg_cost', 'tol'])['ok'].all().unstack('tol').to_string())

---
## Step 5: 실행 로그 저장

In [ ]:
all_runs = dabc_log + sac_log
if all_runs:
    df_log = pd.DataFrame(all_runs)
    log_path = os.path.join(RESULTS_DIR, 'sensitivity_run_log.csv')
    df_log.to_csv(log_path, index=False)
    total, ok_cnt = len(df_log), df_log['ok'].sum()
    print(f'총 {total}건  성공: {ok_cnt}  실패: {total - ok_cnt}')
    failed = df_log[~df_log['ok']]
    if not failed.empty:
        print('\n실패 목록:')
        print(failed.to_string(index=False))
else:
    print('실행된 실험 없음')

---
## Step 6: 결과 수집 및 비교표

학습 완료 후 `results/sensitivity/result_*.json` 파일들을 읽어 DABC vs SAC 비교표를 만듭니다.

In [ ]:
# JSON 결과 파일 수집
result_files = glob.glob(os.path.join(RESULTS_DIR, 'result_*.json'))
if not result_files:
    print(f'결과 파일 없음: {RESULTS_DIR}/result_*.json')
    print('Step 3, 4 학습이 완료되어야 합니다.')
else:
    records = []
    for f in result_files:
        with open(f) as fp:
            records.append(json.load(fp))

    df_all = pd.DataFrame(records)

    # 이번 실험 시나리오 & 알고리즘만 필터
    scen_set = set(ALL_SCENARIOS)
    df_all['scenario'] = list(zip(df_all['deg_cost'], df_all['tol']))
    df_exp = df_all[
        df_all['algo'].isin(['coop_bc', 'coop_sac']) &
        df_all['scenario'].apply(lambda s: s in scen_set)
    ].copy()

    # seed 평균
    df_agg = df_exp.groupby(['algo', 'deg_cost', 'tol']).agg(
        mean_total = ('test_mean_total', 'mean'),
        std_total  = ('test_std_total',  'mean'),
        mean_bid   = ('test_mean_bid',   'mean'),
        mean_ope   = ('test_mean_ope',   'mean'),
        mean_deg   = ('test_mean_deg',   'mean'),
        n_seeds    = ('seed',            'count'),
    ).reset_index()

    # DABC vs SAC pivot
    d = df_agg[df_agg['algo'] == 'coop_bc'][['deg_cost', 'tol', 'mean_total', 'std_total']].copy()
    s = df_agg[df_agg['algo'] == 'coop_sac'][['deg_cost', 'tol', 'mean_total', 'std_total']].copy()
    d = d.rename(columns={'mean_total': 'DABC_mean', 'std_total': 'DABC_std'})
    s = s.rename(columns={'mean_total': 'SAC_mean',  'std_total': 'SAC_std'})

    df_cmp = d.merge(s, on=['deg_cost', 'tol'], how='outer')
    df_cmp['Gap (DABC-SAC)'] = df_cmp['DABC_mean'] - df_cmp['SAC_mean']
    df_cmp['DABC_wins'] = df_cmp['Gap (DABC-SAC)'] > 0

    # Appendix A / B 구분
    df_cmp['Appendix'] = df_cmp.apply(
        lambda r: 'A (deg)' if r['tol'] == 5.0 else 'B (tol)', axis=1
    )

    print('=== DABC vs Coop-SAC 민감도 비교 ===')
    cols = ['Appendix', 'deg_cost', 'tol', 'DABC_mean', 'DABC_std', 'SAC_mean', 'SAC_std', 'Gap (DABC-SAC)', 'DABC_wins']
    print(df_cmp[cols].sort_values(['Appendix', 'deg_cost', 'tol']).to_string(index=False))
    print(f'\nDABC 우세 시나리오: {df_cmp["DABC_wins"].sum()} / {len(df_cmp)}')

---
## Step 7: 시각화 및 CSV 저장

In [ ]:
if 'df_cmp' not in dir():
    print('Step 6을 먼저 실행하세요.')
else:
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    w = 0.35

    # ── Appendix A: Degradation sensitivity (tol=5 MW 고정, deg 변화) ───────
    #    tol==5.0 인 모든 점 → (5,5) baseline 포함
    df_a = df_cmp[df_cmp['tol'] == 5.0].sort_values('deg_cost')
    x = np.arange(len(df_a))
    axes[0].bar(x - w/2, df_a['DABC_mean'], w, yerr=df_a['DABC_std'],
                label='Coop-DABC', color='steelblue', capsize=4)
    axes[0].bar(x + w/2, df_a['SAC_mean'],  w, yerr=df_a['SAC_std'],
                label='Coop-SAC',  color='darkorange', alpha=0.85, capsize=4)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([f'${c}/MWh' for c in df_a['deg_cost']])
    axes[0].set_xlabel('Degradation Cost (tol = 5 MW fixed)')
    axes[0].set_ylabel('Mean Daily Total Revenue ($)')
    axes[0].set_title('Appendix A: Degradation Cost Sensitivity')
    if 5.0 in list(df_a['deg_cost']):
        axes[0].axvline(list(df_a['deg_cost']).index(5.0), ls='--', color='gray',
                        alpha=0.5, label='baseline (5,5)')
    axes[0].legend()

    # ── Appendix B: Tolerance sensitivity (deg=5 $/MWh 고정, tol 변화) ──────
    #    deg_cost==5.0 인 모든 점 → (5,5) baseline 포함
    df_b = df_cmp[df_cmp['deg_cost'] == 5.0].sort_values('tol')
    x = np.arange(len(df_b))
    axes[1].bar(x - w/2, df_b['DABC_mean'], w, yerr=df_b['DABC_std'],
                label='Coop-DABC', color='steelblue', capsize=4)
    axes[1].bar(x + w/2, df_b['SAC_mean'],  w, yerr=df_b['SAC_std'],
                label='Coop-SAC',  color='darkorange', alpha=0.85, capsize=4)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels([f'{t} MW' for t in df_b['tol']])
    axes[1].set_xlabel('Imbalance Tolerance (deg = 5 $/MWh fixed)')
    axes[1].set_ylabel('Mean Daily Total Revenue ($)')
    axes[1].set_title('Appendix B: Imbalance Tolerance Sensitivity')
    if 5.0 in list(df_b['tol']):
        axes[1].axvline(list(df_b['tol']).index(5.0), ls='--', color='gray',
                        alpha=0.5, label='baseline (5,5)')
    axes[1].legend()

    plt.suptitle('Coop-DABC vs Coop-SAC: Parameter Sensitivity', fontsize=13)
    plt.tight_layout()

    fig_path = os.path.join(RESULTS_DIR, 'Sensitivity_DABC_vs_SAC.png')
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.show()

    # CSV 저장
    csv_path = os.path.join(RESULTS_DIR, 'Sensitivity_Summary.csv')
    df_cmp[cols].sort_values(['Appendix', 'deg_cost', 'tol']).to_csv(csv_path, index=False)

    print(f'Figure → {fig_path}')
    print(f'CSV    → {csv_path}')

---
## 참고: 터미널 직접 실행 커맨드

```bash
# (APEN_Major_Revision/code/ 에서)

# ── Appendix A: Degradation sensitivity (tol=5.0 고정) ───────────────
python optimization/run_optimization.py --mode joint --deg_cost 2.5  --tol 5.0
python optimization/run_optimization.py --mode joint --deg_cost 5.0  --tol 5.0
python optimization/run_optimization.py --mode joint --deg_cost 7.5  --tol 5.0
python optimization/run_optimization.py --mode joint --deg_cost 10.0 --tol 5.0

python run_training.py --algo coop_bc  --deg_cost 2.5  --tol 5.0 --seed 0 --expert_csv data/Offline_Expert_Action_joint_deg2.5_tol5.0.csv
python run_training.py --algo coop_sac --deg_cost 2.5  --tol 5.0 --seed 0

# ── Appendix B: Tolerance sensitivity (deg=5.0 고정) ─────────────────
python optimization/run_optimization.py --mode joint --deg_cost 5.0 --tol 2.0
python optimization/run_optimization.py --mode joint --deg_cost 5.0 --tol 10.0

python run_training.py --algo coop_bc  --deg_cost 5.0 --tol 2.0  --seed 0 --expert_csv data/Offline_Expert_Action_joint_deg5.0_tol2.0.csv
python run_training.py --algo coop_sac --deg_cost 5.0 --tol 2.0  --seed 0
```

### 시나리오 요약

| # | deg_cost | tol | Appendix | baseline? |
|---|----------|-----|----------|-----------|
| 1 | 2.5 | 5.0 | A | |
| 2 | **5.0** | **5.0** | A & B | **★** |
| 3 | 7.5 | 5.0 | A | |
| 4 | 10.0 | 5.0 | A | |
| 5 | 5.0 | 2.0 | B | |
| 6 | 5.0 | 10.0 | B | |